# EEG-Conformer SSVEP分类

本 notebook 参考`eegnet.ipynb`的数据格式和预处理，只把EEGNet替换为EEG-Conformer：

```text
EEG [B,1,8,T]
→ 卷积Patch Embedding（时间卷积 + 空间卷积）
→ Transformer Encoder
→ 分类头
→ 15/12/10 Hz
```

- 训练池：`test_2_ssvep`～`test_7_ssvep`
- 验证集：仅从训练池内部按标签分层切分20%
- 最终测试：`test_8_ssvep`
- 所有文件统一截取到共同的最短时间长度
- 均值和标准差只由训练子集计算
- `test_8`不参与训练、归一化、调参或最佳epoch选择


In [1]:
from pathlib import Path
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## 1. 数据加载与训练集归一化

`test_2～test_7`组成训练池，内部切分训练/验证；`test_8`保持完整，只在模型选择结束后做最终测试。


In [2]:
DATA_DIR = Path("data")
TRAIN_FILES = [f"test_{index}_ssvep" for index in range(11, 14)]
TEST_FILES = ["test_15_ssvep"]
FREQUENCIES_HZ = [15, 12, 10]
BATCH_SIZE = 16
MAX_EPOCHS = 200


def load_files(file_names):
    trials, labels = [], []
    for name in file_names:
        x = np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True)
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)
        x = np.stack(x) if np.asarray(x).dtype == object else np.asarray(x)
        if x.ndim != 3 or len(x) != len(y):
            raise ValueError(f"{name}: expected X=[trial,channel,time] with matching labels")
        if not np.isin(y, np.arange(len(FREQUENCIES_HZ))).all():
            raise ValueError(f"{name}: labels must be 0..{len(FREQUENCIES_HZ)-1}")
        trials.append(x)
        labels.append(y)
        print(name, x.shape, y.shape, np.bincount(y, minlength=len(FREQUENCIES_HZ)))
    return trials, labels


train_arrays, train_label_arrays = load_files(TRAIN_FILES)
test_arrays, test_label_arrays = load_files(TEST_FILES)
model_samples = min(x.shape[-1] for x in train_arrays + test_arrays)
train_arrays = [x[:, :, :model_samples] for x in train_arrays]
test_arrays = [x[:, :, :model_samples] for x in test_arrays]

X_pool = np.concatenate(train_arrays).astype(np.float32)
y_pool = np.concatenate(train_label_arrays).astype(np.int64)
X_test = np.concatenate(test_arrays).astype(np.float32)
y_test = np.concatenate(test_label_arrays).astype(np.int64)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_pool, y_pool, test_size=0.20, random_state=SEED, stratify=y_pool
)

# Fit normalization on the training subset only.
mean = X_train.mean(axis=(0, 2), keepdims=True)
std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
X_train = (X_train - mean) / std
X_validation = (X_validation - mean) / std
X_test = (X_test - mean) / std

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_validation = torch.tensor(X_validation, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_validation = torch.tensor(y_validation, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(
    TensorDataset(X_validation, y_validation), batch_size=64, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test), batch_size=64, shuffle=False
)

print("model samples:", model_samples)
print("train / validation / test:", len(y_train), len(y_validation), len(y_test))
print("train batches:", len(train_loader))


test_11_ssvep (61, 8, 373) (61,) [18 25 18]
test_12_ssvep (131, 8, 373) (131,) [41 39 51]
test_13_ssvep (88, 8, 373) (88,) [25 24 39]
test_15_ssvep (85, 8, 373) (85,) [19 36 30]
model samples: 373
train / validation / test: 224 56 85
train batches: 14


## 2. EEG-Conformer

卷积前端先学习时间滤波器和跨8通道的空间滤波器，再把输出沿时间划分为token。Transformer Encoder使用多头自注意力学习token之间的全局关系，最后展平并进行三分类。


In [3]:
class PatchEmbedding(nn.Module):
    def __init__(self, channels, embedding_size=40, temporal_kernel=25, pool_kernel=75, pool_stride=15, dropout=0.5):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Conv2d(
                1, embedding_size, kernel_size=(1, temporal_kernel), bias=False
            ),
            nn.Conv2d(
                embedding_size, embedding_size,
                kernel_size=(channels, 1), bias=False,
            ),
            nn.BatchNorm2d(embedding_size),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool_kernel), stride=(1, pool_stride)),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = self.projection(x)
        return x.squeeze(2).transpose(1, 2)


class EEGConformer(nn.Module):
    def __init__(
        self, channels, samples, classes, embedding_size=40,
        transformer_depth=6, attention_heads=10,
        feedforward_expansion=4, dropout=0.5,
    ):
        super().__init__()
        self.patch_embedding = PatchEmbedding(
            channels=channels, embedding_size=embedding_size, dropout=dropout
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_size,
            nhead=attention_heads,
            dim_feedforward=embedding_size * feedforward_expansion,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=transformer_depth,
            enable_nested_tensor=False,
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, channels, samples)
            tokens = self.patch_embedding(dummy)
            flattened_features = tokens.numel()
            token_count = tokens.shape[1]
        self.token_count = token_count
        self.embedding_size = embedding_size
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_features, 256),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 32),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(32, classes),
        )

    def forward(self, x, return_tokens=False):
        tokens = self.patch_embedding(x)
        tokens = self.transformer(tokens)
        logits = self.classifier(tokens)
        return (logits, tokens) if return_tokens else logits


model = EEGConformer(
    channels=X_train.shape[2],
    samples=X_train.shape[-1],
    classes=len(FREQUENCIES_HZ),
).to(DEVICE)

with torch.no_grad():
    dummy = torch.zeros(2, 1, X_train.shape[2], X_train.shape[-1], device=DEVICE)
    dummy_logits, dummy_tokens = model(dummy, return_tokens=True)
print(model)
print("input shape:", tuple(dummy.shape))
print("token shape:", tuple(dummy_tokens.shape))
print("output shape:", tuple(dummy_logits.shape))
print("trainable parameters:", sum(p.numel() for p in model.parameters()))
assert dummy_logits.shape == (2, len(FREQUENCIES_HZ))


EEGConformer(
  (patch_embedding): PatchEmbedding(
    (projection): Sequential(
      (0): Conv2d(1, 40, kernel_size=(1, 25), stride=(1, 1), bias=False)
      (1): Conv2d(40, 40, kernel_size=(8, 1), stride=(1, 1), bias=False)
      (2): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (3): ELU(alpha=1.0)
      (4): AvgPool2d(kernel_size=(1, 75), stride=(1, 15), padding=0)
      (5): Dropout(p=0.5, inplace=False)
    )
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=40, out_features=40, bias=True)
        )
        (linear1): Linear(in_features=40, out_features=160, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=160, out_features=40, bias=True)
        (norm1): LayerNorm((40,), eps=1e-05, elementwise_affine=True, bias=True)

## 3. 训练与训练池内部验证

使用AdamW、交叉熵和200 epoch。只根据`test_2～test_7`训练池内部的validation选择最佳模型；`test_8`不参与模型选择。


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    predictions, labels = [], []
    for eeg, target in loader:
        logits = model(eeg.to(DEVICE))
        predictions.append(logits.argmax(1).cpu())
        labels.append(target)
    prediction = torch.cat(predictions).numpy()
    label = torch.cat(labels).numpy()
    return float(np.mean(prediction == label)), label, prediction


best_validation_accuracy = -1.0
best_state = None
history = {"train_loss": [], "validation_accuracy": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    for eeg, target in train_loader:
        eeg = eeg.to(DEVICE)
        target = target.to(DEVICE)
        logits = model(eeg)
        loss = criterion(logits, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(target)
    train_loss = loss_sum / len(train_loader.dataset)
    validation_accuracy, _, _ = evaluate(validation_loader)
    history["train_loss"].append(train_loss)
    history["validation_accuracy"].append(validation_accuracy)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch={epoch:03d} train_loss={train_loss:.4f} "
            f"val_acc={validation_accuracy:.4f} "
            f"best_val={best_validation_accuracy:.4f}"
        )

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()
print("best validation accuracy:", best_validation_accuracy)


epoch=001 train_loss=1.2231 val_acc=0.3036 best_val=0.3036
epoch=010 train_loss=0.9133 val_acc=0.5357 best_val=0.5536
epoch=020 train_loss=0.6800 val_acc=0.6429 best_val=0.7143
epoch=030 train_loss=0.3814 val_acc=0.8393 best_val=0.8393
epoch=040 train_loss=0.3639 val_acc=0.8571 best_val=0.9286
epoch=050 train_loss=0.1839 val_acc=0.8750 best_val=0.9286
epoch=060 train_loss=0.2115 val_acc=0.8750 best_val=0.9286
epoch=070 train_loss=0.1756 val_acc=0.8750 best_val=0.9286
epoch=080 train_loss=0.1286 val_acc=0.9464 best_val=0.9464
epoch=090 train_loss=0.1438 val_acc=0.8750 best_val=0.9464
epoch=100 train_loss=0.0942 val_acc=0.8929 best_val=0.9464
epoch=110 train_loss=0.1493 val_acc=0.9107 best_val=0.9464
epoch=120 train_loss=0.1109 val_acc=0.9107 best_val=0.9464
epoch=130 train_loss=0.0566 val_acc=0.8750 best_val=0.9464
epoch=140 train_loss=0.1227 val_acc=0.8036 best_val=0.9464
epoch=150 train_loss=0.0820 val_acc=0.7857 best_val=0.9464
epoch=160 train_loss=0.0394 val_acc=0.9107 best_val=0.94

## 4. `test_8`最终评估

以下结果来自训练池内部validation选出的最佳模型。`test_8`未参与归一化拟合、训练或最佳epoch选择。


In [5]:
test_accuracy, y_true, y_prediction = evaluate(test_loader)
print(f"test_8 accuracy: {test_accuracy:.4f} ({np.sum(y_true == y_prediction)}/{len(y_true)})")
for label, frequency in enumerate(FREQUENCIES_HZ):
    mask = y_true == label
    recall = np.mean(y_prediction[mask] == label)
    print(f"{frequency} Hz recall: {recall:.4f} ({np.sum(y_prediction[mask] == label)}/{np.sum(mask)})")
print("confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]")
print(confusion_matrix(y_true, y_prediction, labels=np.arange(len(FREQUENCIES_HZ))))
print(classification_report(
    y_true, y_prediction,
    labels=np.arange(len(FREQUENCIES_HZ)),
    target_names=[f"{frequency} Hz" for frequency in FREQUENCIES_HZ],
    digits=4,
    zero_division=0,
))


test_8 accuracy: 0.8471 (72/85)
15 Hz recall: 0.8421 (16/19)
12 Hz recall: 0.7222 (26/36)
10 Hz recall: 1.0000 (30/30)
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[16  1  2]
 [ 0 26 10]
 [ 0  0 30]]
              precision    recall  f1-score   support

       15 Hz     1.0000    0.8421    0.9143        19
       12 Hz     0.9630    0.7222    0.8254        36
       10 Hz     0.7143    1.0000    0.8333        30

    accuracy                         0.8471        85
   macro avg     0.8924    0.8548    0.8577        85
weighted avg     0.8835    0.8471    0.8481        85

